In [ ]:
!pip install pyfaidx pyBigWig hictkpy

تحميل ملف الجينوم على البيئة 
واضافة حماية 
"Colab"

In [ ]:

import os
import numpy as np
from pyfaidx import Fasta
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import gc
import time
import shutil
import h5py
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm.auto import tqdm

# تحديد مسار مجلد محلي على هارد الكولاب (Local SSD) لتخزين الكروموسومات المشفرة
DISK_CACHE_DIR = "/content/dna_disk_cache"
os.makedirs(DISK_CACHE_DIR, exist_ok=True)

# مصفوفة البحث السريعة (تغطي A, C, G, T والـ N تأخذ القيمة 4)
_LUT = np.full(256, 4, dtype=np.uint8)
_LUT[ord('A')] = 0; _LUT[ord('C')] = 1
_LUT[ord('G')] = 2; _LUT[ord('T')] = 3

def load_chrom_dna(fasta_path: str, chrom: str) -> np.memmap:
    """
    تقوم بتشفير الكروموسوم وحفظه كملف ثنائي على هارد الكولاب (Disk).
    إذا كان الملف موجوداً مسبقاً، تفتحه فوراً عبر الـ memmap بدون استهلاك للرام.
    """
    # تنظيف اسم الكروموسوم ليتوافق مع الملف
    clean_chrom = chrom.replace("chr", "")
    bin_path = os.path.join(DISK_CACHE_DIR, f"chr{clean_chrom}.bin")
    
    # 1. إذا كان الكروموسوم مشفراً ومحفوظاً مسبقاً على الهارد، افتحيه فوراً
    if os.path.exists(bin_path):
        # نقرأ الطول المخزن للمصفوفة من حجم الملف مباشرة (حيث كل عنصر 1 بايت uint8)
        file_size = os.path.getsize(bin_path)
        return np.memmap(bin_path, dtype=np.uint8, mode='r', shape=(file_size,))

    # 2. إذا لم يكن موجوداً، نقرأه من ملف الـ FASTA الأصلي (أول مرة فقط)
    print(f"💾 [Disk Cache] جاري تشفير وحفظ {chrom} على الهارد...")
    genome = Fasta(fasta_path)
    
    fc = chrom if chrom in genome else (
        chrom.replace("chr", "") if chrom.startswith("chr") else f"chr{chrom}"
    )
    
    # تحويل السلسلة إلى مصفوفة بايتات
    raw = np.frombuffer(str(genome[fc]).upper().encode(), dtype=np.uint8)
    L = len(raw)
    
    # التشفير باستخدام الـ LUT
    encoded = _LUT[raw]
    del genome, raw # تنظيف فوري للرام
    
    # إنشاء ملف الـ memmap على الهارد وسكب البيانات فيه
    fp = np.memmap(bin_path, dtype=np.uint8, mode='w+', shape=(L,))
    fp[:] = encoded[:]
    fp.flush() # التأكد من كتابة البيانات على القرص
    
    # إعادة فتح الملف بوضع القراءة فقط (Read-only) لحمايته
    return np.memmap(bin_path, dtype=np.uint8, mode='r', shape=(L,))

def clear_dna_disk_cache():
    """امسحي الملفات من الهارد لو احتجتِ تفضي مساحة الـ Storage بكولاب."""
    import shutil
    if os.path.exists(DISK_CACHE_DIR):
        shutil.rmtree(DISK_CACHE_DIR)
        os.makedirs(DISK_CACHE_DIR, exist_ok=True)
    print("🗑️ DNA Disk Cache completely cleared from SSD.")

# ══════════════════════════════════════════════════════════════
#  0.  UTILITIES
# ══════════════════════════════════════════════════════════════

def inject_colab_keepalive():
    try:
        from IPython.display import display, Javascript
        js = """
        function clickConnect() {
            try {
                var btn = document.querySelector("colab-connect-button");
                if (btn) btn.click();
            } catch(e) {}
        }
        clickConnect();
        setInterval(clickConnect, 5 * 60 * 1000);
        """
        display(Javascript(js))
        print(" Colab Keepalive مُفعَّل.")
    except Exception as e:
        print(f" Keepalive غير متاح: {e}")


def ensure_genome_local(drive_fasta: str,
                        local_fasta: str = "/content/hg38.fa") -> str:
    if os.path.exists(local_fasta):
        size_gb = os.path.getsize(local_fasta) / 1e9
        print(f" الجينوم موجود محلياً ({size_gb:.1f} GB)")
        return local_fasta
    if not os.path.exists(drive_fasta):
        raise FileNotFoundError(f"ملف الجينوم غير موجود: {drive_fasta}")
    print(f"📋 نسخ الجينوم من Drive → /content ...")
    t0 = time.time()
    shutil.copy2(drive_fasta, local_fasta)
    fai = drive_fasta + ".fai"
    if os.path.exists(fai):
        shutil.copy2(fai, local_fasta + ".fai")
    print(f" تم النسخ في {time.time()-t0:.0f}s")
    return local_fasta


تجهيز ال DATASET

In [ ]:
# ══════════════════════════════════════════════════════════════
#  1.  DATASET
# ══════════════════════════════════════════════════════════════

class ChromogenDataset(Dataset):
    """
    يقرأ HDF5 المنتج من GETDATA.ipynb.

    ما هو مخزون في HDF5:
      hic    : float16, upper-triangle فقط, قيم = gaussian(0.5) + log1p
      dnase  : float16, raw (بدون أي معالجة)
      windows: int32,   [(genomic_start, genomic_end), ...]
      attrs  : resolution=5000, window_size=1280000

    ما نفعله هنا:
      hic   → symmetrize + cast float32  (لا log إضافي — موجود بالفعل)
      dnase → log1p + z-score per-window (normalization يتم هنا)
      dna   → one-hot uint8→float32 من الجينوم
    """

    def __init__(self, h5_path: str, fasta_path: str, chrom: str):
        self.h5_path = h5_path
        self._h5     = None   # lazy per-worker

        with h5py.File(h5_path, "r") as f:
            self.windows     = f["windows"][:]          # (N, 2) int32
            self.resolution  = int(f.attrs["resolution"])   # 5000
            self.window_size = int(f.attrs["window_size"])  # 1280000

            # ── DNase: نحمله كاملاً في RAM (صغير نسبياً) ──────────────
            # raw float16 → float32 في RAM
            self.dnase_full = f["dnase"][:].astype(np.float32)

        self.num_bins = self.window_size // self.resolution  # 256

        # ── DNase global normalization: log1p ──────────────────────────
        self.dnase_full = np.log1p(self.dnase_full)

        # ── DNA: من الكاش المشترك (لا تحميل مكرر لنفس الكروموسوم) ──────
        # لو GM12878/chr1 و K562/chr1 كلاهما يطلبان chr1،
        # التحميل يصير مرة واحدة فقط والباقي يأخذ reference
        self.dna_encoded = load_chrom_dna(fasta_path, chrom)

    # ── Lazy HDF5 per-worker ─────────────────────────────────────────────
    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __del__(self):
        try:
            if self._h5 is not None:
                self._h5.close()
        except Exception:
            pass

    # ── One-hot ─────────────────────────────────────────────────────────
    @staticmethod
    def _decode_onehot(encoded_slice: np.ndarray) -> np.ndarray:
        L   = len(encoded_slice)
        enc = np.zeros((4, L), dtype=np.float32)
        for i in range(4):
            enc[i] = (encoded_slice == i).astype(np.float32)
        unk = encoded_slice == 4
        if unk.any():
            enc[:, unk] = 0.25
        return enc

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        start, end = int(self.windows[idx, 0]), int(self.windows[idx, 1])
        b0 = start // self.resolution
        N  = self.num_bins

        # ── DNA ──────────────────────────────────────────────────────────
        dna_raw = self.dna_encoded[start:end]
        if len(dna_raw) < self.window_size:
            dna_raw = np.pad(dna_raw, (0, self.window_size - len(dna_raw)),
                             constant_values=4)
        dna = torch.from_numpy(self._decode_onehot(dna_raw))   # (4, W)

        # ── DNase: قطع من الكاش + z-score per-window ────────────────────
        dnase_s = self.dnase_full[b0 : b0 + N].copy()
        if len(dnase_s) < N:
            dnase_s = np.pad(dnase_s, (0, N - len(dnase_s)))

        # z-score per-window (يعمل على 256 قيمة فقط، سريع جداً)
        mu  = dnase_s.mean()
        std = dnase_s.std() + 1e-6
        dnase_s = (dnase_s - mu) / std

        dnase = torch.from_numpy(dnase_s)   # (N,)

        # ── Hi-C: upper-triangle → symmetrize ────────────────────────────
        # HDF5 يخزن النص العلوي فقط (gaussian+log1p مطبَّق بالفعل)
        # نعمل: full = upper + upper.T - diag(upper)
        hic_chunk = self._get_h5()["hic"][b0 : b0 + N, b0 : b0 + N].astype(np.float32)

        # عمل Pad إذا كانت الشريحة في آخر الكروموسوم وأصغر من 256
        if hic_chunk.shape[0] < N:
            pad = np.zeros((N, N), dtype=np.float32)
            s   = hic_chunk.shape[0]
            pad[:s, :s] = hic_chunk
            hic_chunk = pad

        # استخلاص المثلث العلوي المحلي للنافذة لضمان نظافة البيانات عند الإزاحات
        hic_upper = np.triu(hic_chunk)
        
        # تحويلها لمصفوفة كاملة متناظرة (Symmetrize)
        hic = hic_upper + hic_upper.T - np.diag(np.diag(hic_upper))

        return {
            "dna":   dna,                        # (4, 1280000)
            "dnase": dnase,                      # (256,)
            "hic":   torch.from_numpy(hic),      # (256, 256)
        }
def build_dataloaders(cell_h5_map: dict,
                      fasta_path: str,
                      val_chroms:  tuple = ("chr18", "chr19"),
                      test_chroms: tuple = ("chr10", "chr21", "chr22"),
                      batch_size:  int   = 2,
                      num_workers: int   = 2):
    train_ds, val_ds, test_ds = [], [], []

    for cell, chrom_map in cell_h5_map.items():
        for chrom, h5_path in chrom_map.items():
            if not os.path.exists(h5_path):
                print(f" مش موجود → {h5_path}")
                continue
            ds = ChromogenDataset(h5_path, fasta_path, chrom)
            if chrom in test_chroms:
                test_ds.append(ds)
            elif chrom in val_chroms:
                val_ds.append(ds)
            else:
                train_ds.append(ds)

    def make_loader(dlist, shuffle):
        if not dlist:
            return None 
        return DataLoader(
            ConcatDataset(dlist),
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=True,
            persistent_workers=(num_workers > 0),
            prefetch_factor=2 if num_workers > 0 else None,
        )

    train_loader = make_loader(train_ds, shuffle=True)
    val_loader   = make_loader(val_ds,   shuffle=False)
    test_loader  = make_loader(test_ds,  shuffle=False)

    print(f" Train: {len(train_loader.dataset)} | "
          f"Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")
    return train_loader, val_loader, test_loader


MODELS 

In [ ]:
# ══════════════════════════════════════════════════════════════
#  2.  MODEL
# ══════════════════════════════════════════════════════════════

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1):
        super().__init__()
        pad = dilation * (kernel - 1) // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, padding=pad,
                      dilation=dilation, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
        )
    def forward(self, x):
        return self.net(x)


class ResBlock(nn.Module):
    def __init__(self, channels, dilation=1):
        super().__init__()
        pad = dilation
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=pad,
                      dilation=dilation, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, 3, padding=pad,
                      dilation=dilation, bias=False),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))

class DNAEncoder(nn.Module):
    """
    Input:  dna (B, 4, W)   W = 1,280,000
            dnase (B, N)    N = 256
    Output: (B, d_model, N)

    التغييرات:
    - DNA يمشي لوحده (4 channels) بدون interpolation للـ DNase
    - اختزال تدريجي: 1,280,000→256,000→51,200→12,800→2,560→256
    - DNase يُدمج كقناة إضافية بعد الاختزال (بدون أي RAM هدر)
    - GroupNorm بدل LayerNorm
    """
    def __init__(self, d_model=128, num_bins=256, window_size=1_280_000):
        super().__init__()
        self.num_bins    = num_bins
        self.window_size = window_size

        # DNA فقط: 4 channels، اختزال تدريجي
        self.stem = nn.Sequential(
            ConvBlock(4, 64, kernel=15),
            nn.MaxPool1d(5),   # 1,280,000 → 256,000
        )
        self.tower = nn.Sequential(
            ResBlock(64,  dilation=1),
            ConvBlock(64, 96, kernel=5), nn.MaxPool1d(5),   # 256,000 → 51,200

            ResBlock(96,  dilation=2),
            ConvBlock(96, 128, kernel=5), nn.MaxPool1d(4),  # 51,200  → 12,800

            ResBlock(128, dilation=4),
            ConvBlock(128, 128, kernel=5), nn.MaxPool1d(5), # 12,800  → 2,560

            ResBlock(128, dilation=8),
            ConvBlock(128, d_model - 1, kernel=3), nn.MaxPool1d(10), # 2,560 → 256
        )

        # GroupNorm على الـ channels مباشرة
        self.out_norm = nn.GroupNorm(1, d_model)

    def forward(self, dna, dnase):
        x = self.stem(dna)           # (B, 64, 256,000)
        x = self.tower(x)            # (B, d_model-1, 256)

        # دمج DNase كقناة إضافية — بدون أي interpolation
        dnase_ch = dnase.unsqueeze(1).float()   # (B, 1, 256)
        x = torch.cat([x, dnase_ch], dim=1)    # (B, d_model, 256)

        x = self.out_norm(x)         # (B, d_model, 256)
        return x


def _build_rope_cache(seq_len: int, head_dim: int, device):
    assert head_dim % 2 == 0
    inv_freq = 1.0 / (10000 ** (
        torch.arange(0, head_dim, 2, device=device).float() / head_dim
    ))
    t     = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    freqs = torch.cat([freqs, freqs], dim=-1)
    return freqs.cos(), freqs.sin()


def _apply_rope(q, k, cos, sin):
    def rotate_half(x):
        h = x.shape[-1] // 2
        return torch.cat([-x[..., h:], x[..., :h]], dim=-1)
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


class FlashRoPEAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int,
                 max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.dropout  = dropout

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out    = nn.Linear(d_model, d_model, bias=False)

        # كودك الأصلي الصحيح تماماً 100%
        cos, sin = _build_rope_cache(max_len, self.head_dim, device=torch.device("cpu"))
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        B, N, D = x.shape

        def split_heads(t):
            return t.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.q_proj(x))
        K = split_heads(self.k_proj(x))
        V = split_heads(self.v_proj(x))

        cos = self.rope_cos[:N]
        sin = self.rope_sin[:N]
        Q, K = _apply_rope(Q, K, cos, sin)

        dropout_p = self.dropout if self.training else 0.0
        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            dropout_p=dropout_p,
            is_causal=False,
        )

        out = out.transpose(1, 2).contiguous().view(B, N, D)
        return self.out(out)

class RoPETransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, max_len=512, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = FlashRoPEAttention(d_model, n_heads, max_len, dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # تطبيق Pre-LN لضمان استقرار تدفق المشتقات في الشبكات العميقة
        x = x + self.attn(self.norm1(x), attn_mask=attn_mask, key_padding_mask=key_padding_mask)
        x = x + self.ff(self.norm2(x))
        return x


class RoPETransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, num_layers, max_len=512, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            RoPETransformerLayer(d_model, n_heads, max_len, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # التمرير عبر الطبقات المتتالية
        for layer in self.layers:
            x = layer(x, attn_mask, key_padding_mask)
        return self.norm(x)


class ChromogenModel(nn.Module):
    
    def __init__(self, d_model=128, nhead=8, num_layers=4,
                 num_bins=256, window_size=1_280_000, dropout=0.1):
        super().__init__()

        self.encoder     = DNAEncoder(d_model, num_bins, window_size)
        self.transformer = RoPETransformerEncoder(
            d_model=d_model, n_heads=nhead,
            num_layers=num_layers,
            max_len=num_bins,
            dropout=dropout,
        )

        # فصل الـ Projection لتطبيق الـ Broadcasting الحقيقي بدون دمج بالذاكرة
        self.pair_proj_i = nn.Linear(d_model, d_model, bias=False)
        self.pair_proj_j = nn.Linear(d_model, d_model, bias=False)
        
        self.pair_norm_act = nn.Sequential(
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        # الـ Decoder ثنائي الأبعاد لقراءة الـ خريطة وتنعيمها
        self.decoder = nn.Sequential(
            nn.Conv2d(d_model, 64, 3, padding=1), nn.GELU(),
            nn.Conv2d(64,      32, 3, padding=1), nn.GELU(),
            nn.Conv2d(32,      16, 3, padding=1), nn.GELU(),
            nn.Conv2d(16,       1, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Conv2d)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, dna, dnase):
        # 1. الـ Encoder والـ Transformer
        z = self.encoder(dna, dnase)    # المخرج أصبح (B, d, N) من التعديل السابق
        z = z.permute(0, 2, 1)          # تحويل إلى (B, N, d) ليناسب الـ Transformer
        z = self.transformer(z)         # المخرج (B, N, d)

        B, N, D = z.shape

        # 2. بناء الـ Pair Representation بالـ Broadcasting الخالص (بدون cat أو expand)
        zi = self.pair_proj_i(z).unsqueeze(2)   # (B, N, 1, D)
        zj = self.pair_proj_j(z).unsqueeze(1)   # (B, 1, N, D)
        
        # الجمع هنا يقوم بعمل الـ Outer Product والـ Broadcasting تلقائياً في الذاكرة
        # pair = zi + zj
        # pair = F.gelu(pair)
        # pair = pair / (pair.norm(dim=-1, keepdim=True) + 1e-6)        # تطبيق الـ Activation والـ Norm
        pair = self.pair_norm_act(zi + zj)
        pair = pair / (pair.norm(dim=-1, keepdim=True) + 1e-6)
        # 3. الـ Decoder ثنائي الأبعاد والتنبؤ
        pair = pair.permute(0, 3, 1, 2)         # (B, D, N, N)
        out  = self.decoder(pair).squeeze(1)    # (B, N, N)

        # 4. إجبار التناظر الرياضي خريطة الـ Hi-C الناتجة
        out = (out + out.transpose(-1, -2)) * 0.5

        return out

LOSS CLass

In [ ]:

# ══════════════════════════════════════════════════════════════
#  3.  LOSS 
# ══════════════════════════════════════════════════════════════

class ChromogenLoss(nn.Module):
    
    def __init__(self, mse_w=0.4, pearson_w=0.4, insulation_w=0.2, window=10):
        super().__init__()
        self.mse_w = mse_w
        self.pea_w = pearson_w
        self.ins_w = insulation_w
        self.window = window

    @staticmethod
    def pearson_loss(pred, target):
        p = pred.reshape(pred.shape[0], -1)
        t = target.reshape(target.shape[0], -1)
        
        pm = p - p.mean(dim=1, keepdim=True)
        tm = t - t.mean(dim=1, keepdim=True)
        
        num = (pm * tm).sum(dim=1)
        den = torch.clamp(pm.norm(dim=1) * tm.norm(dim=1), min=1e-8)
        return (1.0 - (num / den)).mean()

    def insulation_score(self, mat):
        B, N, _ = mat.shape
        w = self.window
        
        # عمل avg_pool على المصفوفة كاملة بضربة واحدة سريعة جداً على الـ GPU
        pooled = F.avg_pool2d(
            mat.unsqueeze(1),  # (B, 1, N, N)
            kernel_size=w, stride=1, padding=0
        ).squeeze(1)           # (B, N-w+1, N-w+1)
        
        scores = torch.diagonal(pooled, dim1=-2, dim2=-1)
        return scores

    def forward(self, pred, target):
        mse = F.mse_loss(pred, target)
        pea = self.pearson_loss(pred, target)
        
        pred_ins = self.insulation_score(pred)
        target_ins = self.insulation_score(target)
        ins_loss = F.mse_loss(pred_ins, target_ins)
        
        total = self.mse_w * mse + self.pea_w * pea + self.ins_w * ins_loss
        return total, {
            "mse":          mse.item(),
            "pearson_loss": pea.item(),
            "pearson_r":    1.0 - pea.item(),
            "insulation":   ins_loss.item(),
        }


Trainer Class

In [ ]:

# ══════════════════════════════════════════════════════════════
#  4.  TRAINER
# ══════════════════════════════════════════════════════════════

class Trainer:
    KEEP_LAST_N = 3

    def __init__(self, model, train_loader, val_loader,
                 lr=3e-4, epochs=50,
                 save_dir="./checkpoints",
                 grad_clip=1.0,
                 grad_accum=4,
                 patience=5,
                 min_delta=1e-4):
        self.device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model        = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.epochs       = epochs
        self.grad_clip    = grad_clip
        self.grad_accum   = grad_accum
        os.makedirs(save_dir, exist_ok=True)
        self.save_dir     = save_dir
        self.patience   = patience
        self.min_delta  = min_delta
        self._es_counter = 0     # كم epoch بدون تحسن
        self._stopped_early = False
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=lr, weight_decay=1e-4,
            betas=(0.9, 0.95),
        )

        self._lr          = lr
        self._total_steps = epochs * math.ceil(len(train_loader) / grad_accum)
        self.scheduler    = None

        self.criterion = ChromogenLoss()
        self.scaler    = torch.amp.GradScaler("cuda")

        self.best_val    = float("inf")
        self.start_epoch = 1
        self.history     = {
            "train_loss": [], "val_loss": [],
            "val_pearson": [], "lr": [],
        }

        self._try_resume()

        completed = (self.start_epoch - 1) * math.ceil(len(train_loader) / grad_accum)
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer, max_lr=lr,
            total_steps=self._total_steps,
            pct_start=0.1,
            last_epoch=completed - 1 if completed > 0 else -1,
        )

    def _try_resume(self):
        epoch_ckpts = sorted([
            f for f in os.listdir(self.save_dir)
            if f.startswith("epoch_") and f.endswith(".pt")
        ])
        if not epoch_ckpts:
            print(" تدريب جديد.")
            return

        latest = os.path.join(self.save_dir, epoch_ckpts[-1])
        print(f"Resume: {latest}")
        try:
            ckpt = torch.load(latest, map_location=self.device)
            self.model.load_state_dict(ckpt["state_dict"])
            self.optimizer.load_state_dict(ckpt["optimizer"])
            self.scaler.load_state_dict(ckpt["scaler"])
            self.start_epoch = ckpt["epoch"] + 1
            self.best_val    = ckpt.get("best_val", float("inf"))
            self.history     = ckpt.get("history", self.history)
            print(f"  Resumed من epoch {ckpt['epoch']} (best_val={self.best_val:.4f})")
        except Exception as e:
            print(f" فشل تحميل الـ checkpoint: {e} → تدريب جديد.")

    def _cleanup_old_ckpts(self):
        epoch_ckpts = sorted([
            f for f in os.listdir(self.save_dir)
            if f.startswith("epoch_") and f.endswith(".pt")
        ])
        for old in epoch_ckpts[:-self.KEEP_LAST_N]:
            try:
                os.remove(os.path.join(self.save_dir, old))
            except Exception:
                pass

    def _save_checkpoint(self, epoch, val_metrics, is_best=False):
        payload = {
            "epoch":      epoch,
            "state_dict": self.model.state_dict(),
            "optimizer":  self.optimizer.state_dict(),
            "scaler":     self.scaler.state_dict(),
            "best_val":   self.best_val,
            "history":    self.history,
            **val_metrics,
        }
        ckpt_path = os.path.join(self.save_dir, f"epoch_{epoch:03d}.pt")
        torch.save(payload, ckpt_path)
        self._cleanup_old_ckpts()

        if is_best:
            torch.save(payload, os.path.join(self.save_dir, "best_model.pt"))
            print(f" Best saved (val={val_metrics.get('val_loss',0):.4f}, "
                  f"r={val_metrics.get('val_pearson',0):.4f})")

    def _run_epoch(self, loader, train=True, epoch=0):
        if len(loader.dataset) == 0:
            return {"loss": 0, "mse": 0, "pearson_r": 0, "insulation": 0}

        self.model.train(train)
        agg = {"loss": 0, "mse": 0, "pearson_r": 0, "insulation": 0}
        n_batches = 0

        phase = "Train" if train else "Val  "
        pbar  = tqdm(loader, desc=f"   Ep {epoch:03d} {phase}",
                     leave=False, ncols=100)

        with torch.set_grad_enabled(train):
            if train:
                self.optimizer.zero_grad()

            for step, batch in enumerate(pbar):
                dna    = batch["dna"].to(self.device,  non_blocking=True)
                dnase  = batch["dnase"].to(self.device, non_blocking=True)
                target = batch["hic"].to(self.device,   non_blocking=True)

                # توسيع نطاق الـ autocast ليشمل الـ forward والـ loss لتسريع الحساب
                with torch.amp.autocast(device_type=self.device.type):
                    pred      = self.model(dna, dnase)
                    loss, sub = self.criterion(pred, target)
                    loss_s    = loss / self.grad_accum

                if train:
                    # الـ backward يستفيد أيضاً من نطاق الـ AMP الموفر للذاكرة
                    self.scaler.scale(loss_s).backward()

                    if (step + 1) % self.grad_accum == 0 or (step + 1) == len(loader):
                        self.scaler.unscale_(self.optimizer)
                        nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                        self.optimizer.zero_grad()
                        self.scheduler.step()

                # تجميع الـ Metrics الحقيقية غير المصغرة للتقرير النهائي
                agg["loss"]       += loss.item()
                agg["mse"]        += sub["mse"]
                agg["pearson_r"]  += sub["pearson_r"]
                agg["insulation"] += sub["insulation"]
                n_batches         += 1

                pbar.set_postfix({
                    "loss": f"{loss.item():.4f}",
                    "r":    f"{sub['pearson_r']:.3f}",
                })

        pbar.close()
        return {k: v / max(n_batches, 1) for k, v in agg.items()}

    def train(self):
        if self.start_epoch > self.epochs:
            print(f" التدريب مكتمل ({self.epochs} epochs).")
            return self.history

        print(f"\n   Chromogen Training")
        print(f"    Device    : {self.device}")
        print(f"    Epochs    : {self.start_epoch}→{self.epochs}")
        print(f"    Grad Acc  : {self.grad_accum}  "
              f"(effective batch = {self.train_loader.batch_size * self.grad_accum})")
        print(f"    Save dir  : {self.save_dir}")
        print("─" * 80)

        for epoch in range(self.start_epoch, self.epochs + 1):
            t0 = time.time()
            tr = self._run_epoch(self.train_loader, train=True,  epoch=epoch)
            vl = self._run_epoch(self.val_loader,   train=False, epoch=epoch)
            dt = time.time() - t0
            lr_now = self.optimizer.param_groups[0]["lr"]

            self.history["train_loss"].append(tr["loss"])
            self.history["val_loss"].append(vl["loss"])
            self.history["val_pearson"].append(vl["pearson_r"])
            self.history["lr"].append(lr_now)

            is_best = vl["loss"] < self.best_val
            if is_best:
                self.best_val = vl["loss"]
                self._es_counter = 0
            else:
                self._es_counter += 1
                if self._es_counter >= self.patience:
                    print(f" Early stopping — {self.patience} epochs بدون تحسن.")
                    self._stopped_early = True

            self._save_checkpoint(
                epoch,
                val_metrics={"val_loss": vl["loss"], "val_pearson": vl["pearson_r"]},
                is_best=is_best,
            )

            remaining = (self.epochs - epoch) * dt
            print(
                f"Ep {epoch:03d}/{self.epochs}  "
                f"Tr={tr['loss']:.4f} (mse={tr['mse']:.4f}, r={tr['pearson_r']:.4f})  "
                f"| Val={vl['loss']:.4f}  r={vl['pearson_r']:.4f}  "
                f"| LR={lr_now:.2e}  [{dt:.0f}s]"
                f"{'  ★' if is_best else ''}"
                f"  (ETA: {remaining/3600:.1f}h)"
            )

            # تنظيف الرام والـ VRAM أولاً بأول لمنع تراكم المخلفات البرمجية والـ OOM
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                if self._stopped_early:
                  break

        print("─" * 80)
        print("   Training complete!")
        return self.history

تجميع استدعاء الـ Pipeline بالكامل

In [ ]:

# ══════════════════════════════════════════════════════════════
#  5.  ENTRY POINT
# ══════════════════════════════════════════════════════════════
def run_training():
    # منع انقطاع الجلسة في Colab
    inject_colab_keepalive()

    DRIVE_FASTA = "/content/drive/MyDrive/Chromogen_Project/data/raw/hg38.fa"
    SAVE_DIR    = "/content/drive/MyDrive/Chromogen_Project/checkpoints"

    # نسخ الجينوم محلياً لتسريع الـ Data Loading ومنع الـ Bottleneck
    FASTA_PATH = ensure_genome_local(
        drive_fasta=DRIVE_FASTA,
        local_fasta="/content/hg38.fa",
    )

    base = "/content/drive/MyDrive/Chromogen_Project/data/processed"

    def h5(cell, split, chrom):
        return f"{base}/{split}/{cell}/{chrom}_hic.h5"

    # خارطة توزيع الكروموسومات الصارمة لمنع الـ Data Leakage
    CELL_H5_MAP = {
        "GM12878": {
            **{c: h5("GM12878", "train", c) for c in [
                "chr1","chr2","chr5","chr8","chr11",
                "chr13","chr14","chr15","chr16","chr17",
            ]},
            **{c: h5("GM12878", "val", c) for c in ["chr18","chr19"]},
            **{c: h5("GM12878", "test", c) for c in ["chr10","chr21","chr22"]},
        },
        "K562": {
            **{c: h5("K562", "train", c) for c in [
                "chr1","chr2","chr5","chr8","chr11",
                "chr13","chr14","chr15","chr16","chr17",
            ]},
            **{c: h5("K562", "val",   c) for c in ["chr18","chr19"]},
            **{c: h5("K562", "test",  c) for c in ["chr10","chr21","chr22"]},
        },
        "HepG2": {
            **{c: h5("HepG2", "train", c) for c in [
                "chr1","chr2","chr5","chr8","chr11",
                "chr13","chr14","chr15","chr16","chr17",
            ]},
            **{c: h5("HepG2", "val",   c) for c in ["chr18","chr19"]},
            **{c: h5("HepG2", "test",  c) for c in ["chr10","chr21","chr22"]},
        },
        "H1-hESC": {
            **{c: h5("H1-hESC", "train", c) for c in [
                "chr1","chr2","chr5","chr8","chr11",
                "chr13","chr14","chr15","chr16","chr17",
            ]},
            **{c: h5("H1-hESC", "val",   c) for c in ["chr18","chr19"]},
            **{c: h5("H1-hESC", "test",  c) for c in ["chr10","chr21","chr22"]},
        },
    }

    # الإعدادات الفوقية (Hyperparameters)
    BATCH_SIZE  = 2
    GRAD_ACCUM  = 4       # Effective batch size = 2 * 4 = 8
    EPOCHS      = 20
    LR          = 3e-4
    NUM_BINS    = 256
    WINDOW_SIZE = 1_280_000
    NUM_WORKERS = 2

    # 1. بناء الـ DataLoaders
    train_loader, val_loader, test_loader = build_dataloaders(
        CELL_H5_MAP, FASTA_PATH,
        val_chroms=("chr18", "chr19"),
        test_chroms=("chr10", "chr21", "chr22"),
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )

    # 2. بناء وتجهيز الـ Model الشامل والموفر للذاكرة
    model = ChromogenModel(
        d_model=128, nhead=8, num_layers=4,
        num_bins=NUM_BINS, window_size=WINDOW_SIZE,
        dropout=0.1,
    )

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f" Total Trainable Parameters: {total_params:,}")

    # 3. تشغيل كلاس التدريب المؤتمت
    trainer = Trainer(
        model, train_loader, val_loader,
        lr=LR, epochs=EPOCHS,
        save_dir=SAVE_DIR,
        grad_accum=GRAD_ACCUM,
        patience=5,    # epochs انتظار قبل الوقف
        min_delta=1e-4,
    )

    history = trainer.train()
    return model, history, test_loader


if __name__ == "__main__":
    model, history, test_loader = run_training()